# 📐 Retrieval Metrics

## What We're Measuring

For each query, we know:
- **Ground truth**: which documents are truly relevant
- **Retrieved**: which documents our system returned

The metrics below measure how well the retrieved set matches the ground truth.

## Metrics We'll Cover

| Metric | Answers the question... |
|---|---|
| **Precision@K** | Of the top K retrieved, what % are relevant? |
| **Recall@K** | Of all relevant docs, what % are in the top K? |
| **F1@K** | Harmonic mean of precision and recall |
| **MRR** | How high did the FIRST relevant doc rank? |
| **NDCG** | Are highly relevant docs ranked at the top? |

In [1]:
# Setup: a small evaluation dataset
# Each entry: query + which doc IDs are truly relevant + what the system retrieved

eval_data = [
    {
        "query": "What is dropout?",
        "relevant": {"doc_1", "doc_2"},          # Ground truth
        "retrieved": ["doc_1", "doc_5", "doc_2", "doc_9", "doc_3"],  # System output (ranked)
    },
    {
        "query": "How does gradient descent work?",
        "relevant": {"doc_10", "doc_11", "doc_12"},
        "retrieved": ["doc_10", "doc_20", "doc_30", "doc_11", "doc_5"],
    },
    {
        "query": "What is transfer learning?",
        "relevant": {"doc_30"},
        "retrieved": ["doc_5", "doc_6", "doc_30", "doc_7", "doc_8"],  # Relevant doc at rank 3
    },
]

print(f"Evaluation set: {len(eval_data)} queries")

Evaluation set: 3 queries


## 1. Precision@K

In [2]:
# Precision@K = (relevant docs in top K) / K

def precision_at_k(retrieved, relevant, k):
    top_k = retrieved[:k]
    hits = sum(1 for doc in top_k if doc in relevant)
    return hits / k


# Calculate for each query
print("Precision@K results:\n")
print(f"{'Query':<35} P@1   P@3   P@5")
print("-" * 55)

for d in eval_data:
    p1 = precision_at_k(d["retrieved"], d["relevant"], 1)
    p3 = precision_at_k(d["retrieved"], d["relevant"], 3)
    p5 = precision_at_k(d["retrieved"], d["relevant"], 5)
    print(f"{d['query']:<35} {p1:.2f}  {p3:.2f}  {p5:.2f}")

print("\n💡 Precision@1 = 1.0 means the top result was always relevant")

Precision@K results:

Query                               P@1   P@3   P@5
-------------------------------------------------------
What is dropout?                    1.00  0.67  0.40
How does gradient descent work?     1.00  0.33  0.40
What is transfer learning?          0.00  0.33  0.20

💡 Precision@1 = 1.0 means the top result was always relevant


## 2. Recall@K

In [3]:
# Recall@K = (relevant docs in top K) / (total relevant docs)

def recall_at_k(retrieved, relevant, k):
    if not relevant:
        return 0.0
    top_k = retrieved[:k]
    hits = sum(1 for doc in top_k if doc in relevant)
    return hits / len(relevant)


print("Recall@K results:\n")
print(f"{'Query':<35} R@1   R@3   R@5")
print("-" * 55)

for d in eval_data:
    r1 = recall_at_k(d["retrieved"], d["relevant"], 1)
    r3 = recall_at_k(d["retrieved"], d["relevant"], 3)
    r5 = recall_at_k(d["retrieved"], d["relevant"], 5)
    print(f"{d['query']:<35} {r1:.2f}  {r3:.2f}  {r5:.2f}")

print("\n💡 Recall@5 = 1.0 means all relevant docs appeared in top 5")

Recall@K results:

Query                               R@1   R@3   R@5
-------------------------------------------------------
What is dropout?                    0.50  1.00  1.00
How does gradient descent work?     0.33  0.33  0.67
What is transfer learning?          0.00  1.00  1.00

💡 Recall@5 = 1.0 means all relevant docs appeared in top 5


## 3. Mean Reciprocal Rank (MRR)

In [4]:
# MRR focuses on rank of the FIRST relevant document
# If first relevant doc is at rank 1 → RR = 1/1 = 1.0
# If first relevant doc is at rank 3 → RR = 1/3 = 0.33
# MRR = average RR across all queries

def reciprocal_rank(retrieved, relevant):
    for rank, doc in enumerate(retrieved, 1):
        if doc in relevant:
            return 1.0 / rank
    return 0.0  # No relevant doc found


def mean_reciprocal_rank(eval_data):
    rrs = [reciprocal_rank(d["retrieved"], d["relevant"]) for d in eval_data]
    return sum(rrs) / len(rrs)


print("Reciprocal Rank per query:\n")
for d in eval_data:
    rr = reciprocal_rank(d["retrieved"], d["relevant"])
    # Find which rank the first relevant doc appeared at
    first_rank = next((i+1 for i, doc in enumerate(d["retrieved"]) if doc in d["relevant"]), None)
    print(f"  '{d['query']}'")
    print(f"   First relevant doc at rank {first_rank} → RR = {rr:.3f}")

mrr = mean_reciprocal_rank(eval_data)
print(f"\nMean Reciprocal Rank (MRR): {mrr:.3f}")
print("  (1.0 = perfect, 0.0 = never found a relevant doc)")

Reciprocal Rank per query:

  'What is dropout?'
   First relevant doc at rank 1 → RR = 1.000
  'How does gradient descent work?'
   First relevant doc at rank 1 → RR = 1.000
  'What is transfer learning?'
   First relevant doc at rank 3 → RR = 0.333

Mean Reciprocal Rank (MRR): 0.778
  (1.0 = perfect, 0.0 = never found a relevant doc)


## 4. NDCG — Normalized Discounted Cumulative Gain

In [5]:
import math

# NDCG rewards relevant docs that appear HIGH in the ranking
# A relevant doc at rank 1 is worth much more than one at rank 5
#
# DCG  = sum of (relevance / log2(rank + 1))
# IDCG = DCG of the ideal (perfect) ranking
# NDCG = DCG / IDCG  (normalized to 0–1)

def dcg_at_k(retrieved, relevant, k):
    dcg = 0.0
    for rank, doc in enumerate(retrieved[:k], 1):
        relevance = 1 if doc in relevant else 0
        dcg += relevance / math.log2(rank + 1)
    return dcg

def ndcg_at_k(retrieved, relevant, k):
    actual_dcg = dcg_at_k(retrieved, relevant, k)
    # Ideal: all relevant docs at the top
    ideal_retrieved = list(relevant) + ["_"] * k
    ideal_dcg = dcg_at_k(ideal_retrieved, relevant, k)
    if ideal_dcg == 0:
        return 0.0
    return actual_dcg / ideal_dcg


print("NDCG@K results:\n")
print(f"{'Query':<35} N@1   N@3   N@5")
print("-" * 55)

for d in eval_data:
    n1 = ndcg_at_k(d["retrieved"], d["relevant"], 1)
    n3 = ndcg_at_k(d["retrieved"], d["relevant"], 3)
    n5 = ndcg_at_k(d["retrieved"], d["relevant"], 5)
    print(f"{d['query']:<35} {n1:.2f}  {n3:.2f}  {n5:.2f}")

NDCG@K results:

Query                               N@1   N@3   N@5
-------------------------------------------------------
What is dropout?                    1.00  0.92  0.92
How does gradient descent work?     1.00  0.47  0.67
What is transfer learning?          0.00  0.50  0.50


## 5. All Metrics Together

In [6]:
# Aggregate all metrics across the evaluation set

K = 5  # Evaluate at K=5

metrics = {
    "Precision@5": [],
    "Recall@5":    [],
    "MRR":         [],
    "NDCG@5":      [],
}

for d in eval_data:
    metrics["Precision@5"].append(precision_at_k(d["retrieved"], d["relevant"], K))
    metrics["Recall@5"].append(recall_at_k(d["retrieved"], d["relevant"], K))
    metrics["MRR"].append(reciprocal_rank(d["retrieved"], d["relevant"]))
    metrics["NDCG@5"].append(ndcg_at_k(d["retrieved"], d["relevant"], K))

print("Retrieval Evaluation Summary")
print("=" * 35)
for metric, values in metrics.items():
    avg = sum(values) / len(values)
    print(f"  {metric:<15}: {avg:.3f}")

print("\n💡 All metrics range from 0.0 (worst) to 1.0 (perfect)")

Retrieval Evaluation Summary
  Precision@5    : 0.333
  Recall@5       : 0.889
  MRR            : 0.778
  NDCG@5         : 0.697

💡 All metrics range from 0.0 (worst) to 1.0 (perfect)


## Key Takeaways 🎯

| Metric | Best for |
|---|---|
| **Precision@K** | When you care about noise (don't want bad docs in context) |
| **Recall@K** | When you can't miss relevant docs (comprehensive search) |
| **MRR** | When you only care that the FIRST result is good |
| **NDCG** | When ranking order within top-K matters |

**Recommended default:** Report **Recall@5** and **NDCG@5** for RAG systems.

---
Next: `02_Generation_Metrics.ipynb`